# Non-Kidney Patient Data
This notebook is analogous to *kidney_patient_data.ipynb* in the following sense: The objective is to generate our dataset for non-kidney patients. Note that we refer to this group as "non-kidney patients" as opposed to "health kidney patients" (or something along those lines) due to the fact that the diagnoses available in MIMIC-IV come from charges made to insurance. Therefore, it is possible for a patient with kidney disease to be admitted to the ICU for a reason unrelated to their kidney problems, and the state of their kidney health would not be captured in the diagnoses records. The steps executed in this notebook are the same as in *kidney_patient_data.ipynb*. However, since we want the kidney health of the non-kidney patients to be as ideal as possible, additional `icd_code` values are used to select patients for the data set. This implies that **not** being included in the kidney disease patient dataset is a necessary but not sufficient condition to be in the non-kidney dataset. Furthermore, due to the significant role of serum creatinine, the non-kidney patients are required to have at least one serum creatinine reading.  

In [1]:
import pandas as pd
import numpy as np

The next two cells load *diagnoses_icd.csv* and *d_icd_diagnoses.csv* in order to extract the `icd_code`s and diagnoses, respectively. 

In [2]:
diagnoses_df = pd.read_csv('../Data/diagnoses_icd.csv')
diagnoses_df.head()

,subject_id,hadm_id,seq_num,icd_code,icd_version
0,10000032,22595853,1,5723,9
1,10000032,22595853,2,78959,9
2,10000032,22595853,3,5715,9
3,10000032,22595853,4,07070,9
4,10000032,22595853,5,496,9


In [3]:
diagnoses_codes_df = pd.read_csv('../Data/d_icd_diagnoses.csv')
diagnoses_codes_df.head()

,icd_code,icd_version,long_title
0,0010,9,Cholera due to vibrio cholerae
1,0011,9,Cholera due to vibrio cholerae el tor
2,0019,9,"Cholera, unspecified"
3,0020,9,Typhoid fever
4,0021,9,Paratyphoid fever A


Next, `icd_code`s associated with diagnoses that have negative implications for kidney health are used to identify the `subject_id` values of patients that we want to avoid when creating the dataset. 

In [4]:
kidney_disease_indices = [2668, 6001, 6002]
kidney_disease_indices += [n for n in range(4660, 4678)]
kidney_disease_indices += [n for n in range(5956, 6000)]
kidney_disease_indices += [n for n in range(6027, 6031)]
kidney_disease_indices += [n for n in range(6086, 6089)]
kidney_disease_indices += [n for n in range(6609, 6614)]
kidney_disease_indices += [n for n in range(37868, 37879)]
kidney_disease_indices += [n for n in range(37860, 37868)]
kidney_disease_indices += [n for n in range(37849, 37860)]
kidney_disease_indices = [n - 2 for n in kidney_disease_indices]
icd_codes = [diagnoses_codes_df.iloc[n]['icd_code'] for n in kidney_disease_indices]
subject_ids = []
for code in icd_codes:
    subject_ids += list(diagnoses_df.loc[diagnoses_df['icd_code'] == code, 'subject_id'])
subject_ids = list(set(subject_ids))

The next cell loads all serum creatinine lab readings. This data will be used to determine the `subject_id` of non-kidney patients.

In [14]:
labs_df = pd.read_csv('../Data/serum_creatinine_labs.csv')
print(labs_df.shape[0])
labs_df.head()

28580


,labevent_id,subject_id,hadm_id,specimen_id,itemid,order_provider_id,charttime,storetime,value,valuenum,valueuom,ref_range_lower,ref_range_upper,flag,priority,comments
0,26,10000032,NaN,83925500,50912,P69FQC,3/23/2180 11:51,3/23/2180 16:40,0.4,0.4,mg/dL,0.4,1.1,NaN,ROUTINE,NaN
1,104,10000032,NaN,95700408,50912,NaN,5/6/2180 22:25,5/6/2180 23:16,0.3,0.3,mg/dL,0.4,1.1,abnormal,STAT,NaN
2,150,10000032,22595853.0,86271148,50912,NaN,5/7/2180 5:05,5/7/2180 7:03,0.3,0.3,mg/dL,0.4,1.1,abnormal,ROUTINE,NaN
3,175,10000032,NaN,19543630,50912,P85UQ1,6/3/2180 12:00,6/3/2180 13:04,0.4,0.4,mg/dL,0.4,1.1,NaN,ROUTINE,NaN
4,187,10000032,NaN,58691952,50912,P69FQC,6/3/2180 12:00,6/3/2180 13:04,0.4,0.4,mg/dL,0.4,1.1,NaN,ROUTINE,NaN


Starting with the serum creatinine lab results loaded above, we now eliminate all rows associated with patients identified above. 

In [15]:
for patient in subject_ids:
    if not labs_df.loc[labs_df['subject_id'] == patient].empty:
        rows_to_drop = labs_df.loc[labs_df['subject_id'] == patient].index
        labs_df = labs_df.drop(rows_to_drop)
print(labs_df.shape[0])

11974


The final step prior to extracting all lab results is eliminating patients with no diagnoses. The motivation behind this step is that we want to have some idea about the medical history of the patients in our dataset. 

In [16]:
normal_kidney_ids = list(set(labs_df['subject_id']))
for patient in normal_kidney_ids:
    if diagnoses_df.loc[diagnoses_df['subject_id'] == patient].empty:
        rows_to_drop = labs_df.loc[labs_df['subject_id'] == patient].index
        labs_df = labs_df.drop(rows_to_drop)
print(labs_df.shape[0])
normal_kidney_ids = list(set(labs_df['subject_id']))

10760


Finally, the lab data for the remaining patients is extracted from *labevents.csv*. The result is saved as *kidney_all_labs.csv.gz*.

In [21]:
labevents_cols = ['labevent_id',
                 'subject_id',
                 'hadm_id',
                 'specimen_id',
                 'itemid',
                 'order_provider_id',
                 'charttime',
                 'storetime',
                 'value',
                 'valuenum',
                 'valueuom',
                 'ref_range_lower',
                 'ref_range_upper',
                 'flag',
                 'priority',
                 'comments']
normal_kidney_labs_df = pd.DataFrame(columns = labevents_cols, dtype = object)
chunk_size = 10**6
for n, chunk in enumerate(pd.read_csv('../Data/labevents.csv.gz', compression = 'gzip', chunksize = chunk_size)):
    print(n)
    for patient in normal_kidney_ids:
        patient_chunk = chunk.loc[chunk['subject_id'] == patient]
        normal_kidney_labs_df = pd.concat([normal_kidney_labs_df, patient_chunk])


0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158


In [23]:
normal_kidney_labs_df.to_csv('../Data/kidney_all_labs.csv.gz', index = False, compression='gzip',)